In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [2]:
import pandas as pd
df = pd.read_csv('amazon_review.csv')

In [3]:
df.head()

,comments,rating,title
0,Very slow and not good 8gb ram according spee...,4.0,Very solow speed and hang laptop bad
1,Really hate the product. Its wastage of money....,1.0,Its a regret!!
2,The Product Worth it,5.0,Good Product
3,Your browser does not support HTML5 video.\r\n...,1.0,Please don't buy
4,This is a review after more than 6 months of ...,2.0,Not good


In [4]:
len(df)

511

In [5]:
# Keep only laptop-related product reviews
pattern = r"\b(laptop|notebook|macbook|chromebook|dell|hp|lenovo|asus|acer|msi|surface|thinkpad|vivobook|ideapad|mac)\b"

df = df[df['title'].str.contains(pattern, case=False, na=False)]

print("After filtering laptop reviews:", len(df))
df.head()


After filtering laptop reviews: 106


C:\Users\basav\AppData\Local\Temp\ipykernel_24200\1969864240.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[df['title'].str.contains(pattern, case=False, na=False)]


,comments,rating,title
0,Very slow and not good 8gb ram according spee...,4.0,Very solow speed and hang laptop bad
5,"I purchased GR0011AU (1TB HDD no SSD, Ryzen 3 ...",4.0,"Decent laptop, good customer support during setup"
8,laptop is hanging if we install more products,1.0,laptop is hanging if we install more products
14,I bought recently and was not disappointed wit...,5.0,"Good laptop for students, affordable."
15,I bought this laptop with 8GB of ram and 1TB H...,5.0,HP 15s-gr0011au


In [6]:
df.shape

(106, 3)

In [7]:
df['text'] = (df['title'].fillna('') + ' ' + df['comments'].fillna('')).str.strip()
df = df[df['text'].str.len() > 0]   # drop empty text rows


In [8]:
def rating_to_label(r):
    if r >= 4:
        return 1      # positive
    elif r <= 2:
        return 0      # negative
    else:
        return None   # neutral (we drop for binary model)

df['label'] = df['rating'].apply(rating_to_label)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

df[['rating', 'label', 'text']].head()
print(df['label'].value_counts(normalize=True))


label
1    0.625
0    0.375
Name: proportion, dtype: float64


In [9]:
# Cell 6: Cleaning function
import string

def clean_text(t: str) -> str:
    t = str(t).lower()
    # remove non-alphanumeric
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    # collapse spaces
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df['clean_text'] = df['text'].apply(clean_text)
df[['clean_text']].head()


,clean_text
0,very solow speed and hang laptop bad very slow...
5,decent laptop good customer support during set...
8,laptop is hanging if we install more products ...
14,good laptop for students affordable i bought r...
15,hp 15s gr0011au i bought this laptop with 8gb ...


In [10]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [11]:
print(len(X_train)) 
print(len(X_test))

76
20


In [12]:
# Cell 8: Basic model - CountVectorizer + MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

count_vec = CountVectorizer(max_features=5000)
X_train_vec = count_vec.fit_transform(X_train)
X_test_vec = count_vec.transform(X_test)

nb_clf = MultinomialNB()
nb_clf.fit(X_train_vec, y_train)

y_pred_nb = nb_clf.predict(X_test_vec)

print("=== BASIC MODEL (CountVectorizer + NaiveBayes) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification report:\n", classification_report(y_test, y_pred_nb))


=== BASIC MODEL (CountVectorizer + NaiveBayes) ===
Accuracy: 0.9

Classification report:
               precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.90      0.90      0.90        20



In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

log_reg = LogisticRegression(max_iter=2000, class_weight="balanced")
log_reg.fit(X_train_tfidf, y_train)

y_pred_lr = log_reg.predict(X_test_tfidf)
print("=== INTERMEDIATE MODEL (TF-IDF + Logistic Regression) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


=== INTERMEDIATE MODEL (TF-IDF + Logistic Regression) ===
Accuracy: 0.9
              precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.90      0.90      0.90        20



In [14]:
from sklearn.metrics import classification_report, confusion_matrix

print("BASIC")
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

print("INTERMEDIATE")
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))


BASIC
              precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.90      0.90      0.90        20

[[ 7  1]
 [ 1 11]]
INTERMEDIATE
              precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.90      0.90      0.90        20

[[ 7  1]
 [ 1 11]]
              precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg     

In [15]:
import pickle, os

os.makedirs("models", exist_ok=True)

with open("models/sentiment_model.pkl", "wb") as f:
    pickle.dump(log_reg, f)

with open("models/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Saved models/sentiment_model.pkl and models/tfidf.pkl")


Saved models/sentiment_model.pkl and models/tfidf.pkl


In [16]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments


c:\Users\basav\PYTHON_PRACTICE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=256)


In [18]:
%pip install transformers torch datasets


Note: you may need to restart the kernel to use updated packages.


In [19]:
from datasets import Dataset

In [20]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)


Map: 100%|██████████| 20/20 [00:00<00:00, 2501.97 examples/s]


In [21]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
%pip install accelerate>=0.26.0

Note: you may need to restart the kernel to use updated packages.


In [23]:
training_args = TrainingArguments(
    output_dir='./bert_model',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,No log,0.631525
2,No log,0.607408
3,No log,0.593376


TrainOutput(global_step=15, training_loss=0.6108646392822266, metrics={'train_runtime': 74.7471, 'train_samples_per_second': 3.05, 'train_steps_per_second': 0.201, 'total_flos': 12800697296688.0, 'train_loss': 0.6108646392822266, 'epoch': 3.0})

In [25]:
results = trainer.evaluate()
results


{'eval_loss': 0.5933762192726135,
 'eval_runtime': 1.4632,
 'eval_samples_per_second': 13.668,
 'eval_steps_per_second': 1.367,
 'epoch': 3.0}

In [26]:
model.save_pretrained("models/bert_sentiment")
tokenizer.save_pretrained("models/bert_sentiment")


('models/bert_sentiment\\tokenizer_config.json',
 'models/bert_sentiment\\special_tokens_map.json',
 'models/bert_sentiment\\vocab.txt',
 'models/bert_sentiment\\added_tokens.json',
 'models/bert_sentiment\\tokenizer.json')

In [27]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    return {"Negative": float(probs[0][0]), "Positive": float(probs[0][1])}

samples = [
    "Battery life is terrible and it overheats",
    "Amazing display and great performance!"
]

for s in samples:
    print(s, "->", predict_sentiment(s))


Battery life is terrible and it overheats -> {'Negative': 0.42664575576782227, 'Positive': 0.5733542442321777}
Amazing display and great performance! -> {'Negative': 0.3545435667037964, 'Positive': 0.6454563736915588}


In [28]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }



In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)


In [30]:
trainer.train()
metrics = trainer.evaluate()
metrics


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.491766,0.750000,0.733333,0.916667,0.814815
2,No log,0.434529,0.800000,0.785714,0.916667,0.846154
3,No log,0.401011,0.900000,0.916667,0.916667,0.916667


{'eval_loss': 0.4010106027126312,
 'eval_accuracy': 0.9,
 'eval_precision': 0.9166666666666666,
 'eval_recall': 0.9166666666666666,
 'eval_f1': 0.9166666666666666,
 'eval_runtime': 1.605,
 'eval_samples_per_second': 12.461,
 'eval_steps_per_second': 1.246,
 'epoch': 3.0}

In [31]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    return {"Negative": float(probs[0][0]), "Positive": float(probs[0][1])}

samples = [
    "Battery life is terrible and it overheats",
    "Amazing display and great performance!"
]

for s in samples:
    print(s, "->", predict_sentiment(s))


Battery life is terrible and it overheats -> {'Negative': 0.556509792804718, 'Positive': 0.44349023699760437}
Amazing display and great performance! -> {'Negative': 0.2672010064125061, 'Positive': 0.7327989935874939}


In [32]:
# Cell 12: Quick manual test
test_texts = [
    "This laptop is amazing, very fast and great battery life",
    "Worst product ever, it keeps hanging and is extremely slow",
    "Average performance, nothing special but okay for the price"
]

cleaned = [clean_text(t) for t in test_texts]
X_custom = tfidf.transform(cleaned)
probs = log_reg.predict_proba(X_custom)[:, 1]
preds = log_reg.predict(X_custom)

for txt, p, prob in zip(test_texts, preds, probs):
    label = "Positive" if p == 1 else "Negative"
    print(f"Review: {txt}")
    print(f"Predicted: {label} (positive_prob={prob:.3f})")
    print("-"*60)


Review: This laptop is amazing, very fast and great battery life
Predicted: Positive (positive_prob=0.559)
------------------------------------------------------------
Review: Worst product ever, it keeps hanging and is extremely slow
Predicted: Negative (positive_prob=0.394)
------------------------------------------------------------
Review: Average performance, nothing special but okay for the price
Predicted: Positive (positive_prob=0.561)
------------------------------------------------------------


In [33]:
# After cleaning + labeling:
X = df['clean_text']
y = df['label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [34]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svc = SVC(
    kernel='linear',          # good for text data
    probability=True,         # so we can use predict_proba
    class_weight='balanced',  # helps with slight imbalance
    random_state=42
)

svc.fit(X_train_tfidf, y_train)

y_pred_svc = svc.predict(X_test_tfidf)

print("=== SVC (TF-IDF) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svc))
print("\nClassification report:\n", classification_report(y_test, y_pred_svc))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_svc))


=== SVC (TF-IDF) ===
Accuracy: 0.9

Classification report:
               precision    recall  f1-score   support

           0       0.88      0.88      0.88         8
           1       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.90      0.90      0.90        20

Confusion matrix:
 [[ 7  1]
 [ 1 11]]


In [35]:
import pickle, os

os.makedirs("models", exist_ok=True)

with open("models/sentiment_model.pkl", "wb") as f:
    pickle.dump(svc, f)

with open("models/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Saved SVC sentiment_model.pkl and tfidf.pkl")


Saved SVC sentiment_model.pkl and tfidf.pkl


In [36]:
# Cell 12: Quick manual test
test_texts = [
    "This laptop is amazing, very fast and great battery life",
    "Worst product ever, it keeps hanging and is extremely slow",
    "Average performance, nothing special but okay for the price"
]

cleaned = [clean_text(t) for t in test_texts]
X_custom = tfidf.transform(cleaned)
probs = svc.predict_proba(X_custom)[:, 1]
preds = svc.predict(X_custom)

for txt, p, prob in zip(test_texts, preds, probs):
    label = "Positive" if p == 1 else "Negative"
    print(f"Review: {txt}")
    print(f"Predicted: {label} (positive_prob={prob:.3f})")
    print("-"*60)


Review: This laptop is amazing, very fast and great battery life
Predicted: Positive (positive_prob=0.935)
------------------------------------------------------------
Review: Worst product ever, it keeps hanging and is extremely slow
Predicted: Negative (positive_prob=0.001)
------------------------------------------------------------
Review: Average performance, nothing special but okay for the price
Predicted: Positive (positive_prob=0.833)
------------------------------------------------------------
